In [2]:
import pandas as pd
import numpy as np

# CONFIG
INPUT_FILE = "../data stuff/augmented_telemetry_filtered.csv"
OUTPUT_FILE = "../data stuff/augmented_telemetry_filtered_with_location.csv"

MAX_DIST_M = 1000  # 1 km

# Communities (lat, lon)
COMMUNITIES = np.array([
    (15.5007, 32.5599), (15.6133, 32.5322), (14.4015, 33.5198), 
    (13.1747, 30.2097), (12.8628, 32.9838), (14.0000, 31.0000),
    (15.0000, 35.0000), (13.5000, 34.0000), (12.5000, 30.5000),
    (15.8000, 33.2000), (14.2000, 32.1000), (13.9000, 35.5000),
    (14.8000, 34.5000), (15.2000, 31.8000), (12.9000, 33.9000),
    (13.2000, 31.2000), (14.5000, 33.1000), (15.4000, 32.8000),
    (13.7000, 30.8000)
])

COMM_IDS = [f"loc_{i+1:03d}" for i in range(len(COMMUNITIES))]

# Helper: Haversine distance (meters)
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

# LOAD DATA
df = pd.read_csv(INPUT_FILE)
df["lat"] = df["lat"].astype(float)
df["lon"] = df["lon"].astype(float)

# VECTORIZE NEAREST COMMUNITY ASSIGNMENT
lats = df["lat"].values[:, None]       # shape (n,1)
lons = df["lon"].values[:, None]       # shape (n,1)
comm_lats = COMMUNITIES[:,0][None,:]   # shape (1,m)
comm_lons = COMMUNITIES[:,1][None,:]   # shape (1,m)

# distances: shape (n_points, n_communities)
distances = haversine_vectorized(lats, lons, comm_lats, comm_lons)

# nearest community
nearest_idx = np.argmin(distances, axis=1)
df["location_id"] = [COMM_IDS[i] for i in nearest_idx]

# optional: flag if truly within MAX_DIST_M
df["in_community"] = distances[np.arange(len(df)), nearest_idx] <= MAX_DIST_M

# SAVE BACK TO TELEMETRY CSV
df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved augmented telemetry with location_id to {OUTPUT_FILE}")
print(df[["lat","lon","location_id","in_community"]].head(10))


Saved augmented telemetry with location_id to ../data stuff/augmented_telemetry_filtered_with_location.csv
        lat       lon location_id  in_community
0  14.39523  33.52871     loc_003         False
1  13.89125  35.49622     loc_012         False
2  15.60635  33.91738     loc_010         False
3  15.39167  32.80353     loc_018         False
4  14.00292  31.00291     loc_006          True
5  14.40914  33.51272     loc_003         False
6  14.39915  33.51229     loc_003          True
7  14.50740  33.10841     loc_017         False
8  12.90540  33.89572     loc_015          True
9  15.10040  33.60577     loc_003         False
